# StackKernels Tesla T4 Validation Suite

Comprehensive validation of the `stackkernels._opt` optimization layer on a real NVIDIA Tesla T4 GPU.

In [20]:
# Cell 1: Check Connection & Hardware
!nvidia-smi

Fri Oct 30 09:26:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P0             28W /   70W |     437MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [21]:
# Cell 2a: Environment Versions
import sys
import torch
import triton

print("=== Environment Info ===")
print("Python version: ", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("Triton version: ", triton.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device count:   ", torch.cuda.device_count())
    print("Device name:    ", torch.cuda.get_device_name(0))
    print("Compute cap:    ", torch.cuda.get_device_capability(0))
    props = torch.cuda.get_device_properties(0)
    print(f"SM Count:        {getattr(props, 'multi_processor_count', 'unknown')}")
    print(f"Total Memory:    {props.total_memory / (1024**3):.2f} GB")

=== Environment Info ===
Python version:  3.13.16
PyTorch version: 2.11.0+cu130
Triton version:  3.6.0
CUDA available:  True
Device count:    1
Device name:     Tesla T4
Compute cap:     (7, 5)
SM Count:        40
Total Memory:    14.56 GB


In [22]:
# Cell 3: Repository Setup & Installation
import os
import sys

# Check if running inside Colab /content and clone if needed
if not os.path.exists("src/stackkernels"):
    if os.path.exists("/content/Stackkernels"):
        os.chdir("/content/Stackkernels")
    elif os.path.exists("Stackkernels"):
        os.chdir("Stackkernels")
    else:
        !git clone https://github.com/stackformer-labs/Stackkernels.git /content/Stackkernels
        os.chdir("/content/Stackkernels")

print("Working directory:", os.getcwd())
!pip install -e . --no-deps
!pip install pytest

# Ensure src is immediately importable in current Python process
src_dir = os.path.abspath("src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import stackkernels
print("Successfully loaded stackkernels from:", stackkernels.__file__)

Working directory: /content/Stackkernels
Obtaining file:///content/Stackkernels
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for stackkernels (pyproject.toml) ... done
  Created wheel for stackkernels: filename=stackkernels-0.1.0-py3-none-any.whl size=2385 sha256=a911af54a2456e5a3e69186cbaea61c64ec41d83f5dd0fafd4ece09e5d20eb13
  Stored in directory: /tmp/pip-ephem-wheel-cache-ut0in58y/wheels/f2/f5/5b/3579bbaf28d0794f8c4b2a4771ec462f7a282492598830adbe
Successfully built stackkernels
  Attempting uninstall: stackkernels
    Found existing installation: stackkernels 0.1.0
    Uninstalling stackkernels-0.1.0:
      Successfully uninstalled stackkernels-0.1.0
Successfully loaded stackkernels from: /content/Stackkernels/src/stackkernels/__init__.py


In [23]:
# Cell 4: Check 0 - Rule Audit
print("=== Check 0: Rule Audit ===")
print("1. Checking for @triton.autotune in src (must be empty):")
!git grep -n "triton.autotune" -- src

print("\n2. Checking _opt/heuristics and _opt/classes for framework imports (must be empty):")
!git grep -nE "^(import|from) +(torch|jax|triton)" -- src/stackkernels/_opt/heuristics src/stackkernels/_opt/classes

print("\n3. Checking _core for _opt imports (must be empty or reported):")
!git grep -n "_opt" -- src/stackkernels/_core

print("\n4. Checking diff on src/stackkernels/_core since 6584a27~1:")
!git diff 6584a27~1 --stat -- src/stackkernels/_core

=== Check 0: Rule Audit ===
1. Checking for @triton.autotune in src (must be empty):

2. Checking _opt/heuristics and _opt/classes for framework imports (must be empty):
src/stackkernels/_opt/heuristics/_util.py:3:import triton

3. Checking _core for _opt imports (must be empty or reported):
src/stackkernels/_core/activation/elementwise.py:11:from stackkernels._opt.heuristics._util import determine_num_warps
src/stackkernels/_core/activation/glu.py:11:from stackkernels._opt.heuristics._util import determine_num_warps
src/stackkernels/_core/attention/decode.py:10:from stackkernels._opt.heuristics.flash_decode import heuristic as decode_heuristic
src/stackkernels/_core/attention/decode.py:11:from stackkernels._opt.memory.layout import get_padded_head_dim, validate_attention_layout
src/stackkernels/_core/attention/decode.py:12:from stackkernels._opt.memory.workspace import allocate_split_kv_buffers
src/stackkernels/_core/attention/flash_bwd.py:10:from stackkernels._opt.heuristics._util im

In [24]:
# Cell 5: Check 2b - Kernel Status Matrix across all ops
import os, sys
if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))

import torch
import stackkernels
from stackkernels._options import options

options(backend="triton")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Backend forced to: {options().backend}, Running on: {device}")

# Import references
import stackkernels._reference.torch as ref
import stackkernels.torch as s_torch

results = []

def eval_op(op_name, fn_torch, fn_ref, input_fn, has_bwd=True):
    for dt_name, dt in [("fp16", torch.float16), ("fp32", torch.float32)]:
        if fn_torch is None:
            results.append((op_name, dt_name, "No", "N/A", "N/A", "N/A", "Stub / Not implemented"))
            continue
        try:
            args_t, args_r = input_fn(dt, device)
            # Forward
            out_t = fn_torch(*args_t)
            out_r = fn_ref(*args_r)
            # Unpack if tuple
            val_t = out_t[0] if isinstance(out_t, tuple) else out_t
            val_r = out_r[0] if isinstance(out_r, tuple) else out_r
            
            max_abs = (val_t.float() - val_r.float()).abs().max().item()
            max_rel = ((val_t.float() - val_r.float()).abs() / (val_r.float().abs() + 1e-5)).max().item()
            err_str = f"{max_abs:.2e} / {max_rel:.2e}"
            
            bwd_str = "no backward"
            if has_bwd:
                try:
                    loss_t = val_t.sum()
                    loss_t.backward()
                    loss_r = val_r.sum()
                    loss_r.backward()
                    bwd_str = "yes (matches)"
                except Exception as eb:
                    bwd_str = f"failed: {str(eb)[:15]}"
            
            results.append((op_name, dt_name, "Yes", "Yes", err_str, bwd_str, "OK"))
        except Exception as e:
            results.append((op_name, dt_name, "Yes", "No", "N/A", "N/A", f"Error: {str(e)[:30]}"))

# 1. Norms
def norm_inp(dt, dev):
    x1 = torch.randn(2, 64, device=dev, dtype=dt, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    w1 = torch.randn(64, device=dev, dtype=dt, requires_grad=True)
    w2 = w1.detach().clone().requires_grad_(True)
    return (x1, w1), (x2, w2)

eval_op("rms_norm", getattr(s_torch, "rms_norm", None), getattr(ref, "rms_norm", None), norm_inp)

def lnorm_inp(dt, dev):
    x1 = torch.randn(2, 64, device=dev, dtype=dt, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    w1 = torch.randn(64, device=dev, dtype=dt, requires_grad=True)
    w2 = w1.detach().clone().requires_grad_(True)
    b1 = torch.randn(64, device=dev, dtype=dt, requires_grad=True)
    b2 = b1.detach().clone().requires_grad_(True)
    return (x1, w1, b1), (x2, w2, b2)

eval_op("layer_norm", getattr(s_torch, "layer_norm", None), getattr(ref, "layer_norm", None), lnorm_inp)

def fanorm_inp(dt, dev):
    x1 = torch.randn(2, 64, device=dev, dtype=dt, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    r1 = torch.randn(2, 64, device=dev, dtype=dt, requires_grad=True)
    r2 = r1.detach().clone().requires_grad_(True)
    w1 = torch.randn(64, device=dev, dtype=dt, requires_grad=True)
    w2 = w1.detach().clone().requires_grad_(True)
    return (x1, r1, w1), (x2, r2, w2)

eval_op("fused_add_norm", getattr(s_torch, "fused_add_norm", None), getattr(ref, "fused_add_norm", None), fanorm_inp)

# 2. Activations
def act_inp(dt, dev):
    x1 = torch.randn(2, 64, device=dev, dtype=dt, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    return (x1,), (x2,)

def glu_inp(dt, dev):
    x1 = torch.randn(2, 128, device=dev, dtype=dt, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    return (x1,), (x2,)

eval_op("swiglu", getattr(s_torch, "swiglu", None), getattr(ref, "swiglu", None), glu_inp)
eval_op("geglu", getattr(s_torch, "geglu", None), getattr(ref, "geglu", None), glu_inp)
for act in ["gelu", "silu", "relu", "leaky_relu", "sigmoid"]:
    eval_op(act, getattr(s_torch, act, None), getattr(ref, act, None), act_inp)

# 3. RoPE
def rope_inp(dt, dev):
    q1 = torch.randn(2, 4, 16, 64, device=dev, dtype=dt, requires_grad=True)
    q2 = q1.detach().clone().requires_grad_(True)
    cos, sin = ref.precompute_freqs(64, 16, device=dev)
    cos = cos.to(dt)
    sin = sin.to(dt)
    return (q1, cos, sin), (q2, cos, sin)

eval_op("rope", getattr(s_torch, "apply_rope", None), getattr(ref, "apply_rope", None), rope_inp)

# 4. Loss
def ce_inp(dt, dev):
    logits1 = torch.randn(4, 128, device=dev, dtype=dt, requires_grad=True)
    logits2 = logits1.detach().clone().requires_grad_(True)
    targets = torch.randint(0, 128, (4,), device=dev)
    return (logits1, targets), (logits2, targets)

eval_op("cross_entropy", getattr(s_torch, "cross_entropy", None), getattr(ref, "cross_entropy", None), ce_inp)

def flce_inp(dt, dev):
    x1 = torch.randn(4, 32, device=dev, dtype=dt, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    w1 = torch.randn(64, 32, device=dev, dtype=dt, requires_grad=True)
    w2 = w1.detach().clone().requires_grad_(True)
    targets = torch.randint(0, 64, (4,), device=dev)
    return (x1, w1, targets), (x2, w2, targets)

eval_op("fused_linear_ce", getattr(s_torch, "fused_linear_cross_entropy", None), getattr(ref, "fused_linear_cross_entropy", None), flce_inp)

# 5. Attention
def attn_inp(dt, dev):
    q1 = torch.randn(2, 4, 32, 64, device=dev, dtype=dt, requires_grad=True)
    k1 = torch.randn(2, 4, 32, 64, device=dev, dtype=dt, requires_grad=True)
    v1 = torch.randn(2, 4, 32, 64, device=dev, dtype=dt, requires_grad=True)
    q2 = q1.detach().clone().requires_grad_(True)
    k2 = k1.detach().clone().requires_grad_(True)
    v2 = v1.detach().clone().requires_grad_(True)
    return (q1, k1, v1), (q2, k2, v2)

eval_op("flash_attention", getattr(s_torch, "flash_attention", None), getattr(ref, "attention_reference", None), attn_inp)

def dec_inp(dt, dev):
    q1 = torch.randn(2, 4, 1, 64, device=dev, dtype=dt)
    k1 = torch.randn(2, 4, 32, 64, device=dev, dtype=dt)
    v1 = torch.randn(2, 4, 32, 64, device=dev, dtype=dt)
    q2 = q1.detach().clone()
    k2 = k1.detach().clone()
    v2 = v1.detach().clone()
    return (q1, k1, v1), (q2, k2, v2)

eval_op("flash_decode", getattr(s_torch, "flash_decode", None), getattr(ref, "attention_reference", None), dec_inp, has_bwd=False)

# 6. Unimplemented stubs
eval_op("dropout", getattr(s_torch, "dropout", None), None, None)
eval_op("fused_dropout_add", getattr(s_torch, "fused_dropout_add", None), None, None)
eval_op("kv_update", getattr(s_torch, "kv_update", None), None, None)
eval_op("topk_topp_sampling", getattr(s_torch, "topk_topp_sampling", None), None, None)

# Format and print table
print(f"\n{'Op':<22} | {'Dtype':<5} | {'Impl?':<5} | {'Fwd Triton':<10} | {'MaxAbs / RelErr':<17} | {'Bwd Triton':<14} | {'Status'}")
print("-" * 95)
for r in results:
    print(f"{r[0]:<22} | {r[1]:<5} | {r[2]:<5} | {r[3]:<10} | {r[4]:<17} | {r[5]:<14} | {r[6]}")

Backend forced to: triton, Running on: cuda

Op                     | Dtype | Impl? | Fwd Triton | MaxAbs / RelErr   | Bwd Triton     | Status
-----------------------------------------------------------------------------------------------
rms_norm               | fp16  | Yes   | Yes        | 3.91e-03 / 9.57e-04 | yes (matches)  | OK
rms_norm               | fp32  | Yes   | Yes        | 0.00e+00 / 0.00e+00 | yes (matches)  | OK
layer_norm             | fp16  | Yes   | Yes        | 1.95e-03 / 5.80e-03 | yes (matches)  | OK
layer_norm             | fp32  | Yes   | Yes        | 4.77e-07 / 3.95e-06 | yes (matches)  | OK
fused_add_norm         | fp16  | Yes   | Yes        | 3.91e-03 / 9.73e-04 | yes (matches)  | OK
fused_add_norm         | fp32  | Yes   | Yes        | 0.00e+00 / 0.00e+00 | yes (matches)  | OK
swiglu                 | fp16  | Yes   | No         | N/A               | N/A            | Error: swiglu() missing 1 required po
swiglu                 | fp32  | Yes   | No         | N/

In [25]:
# Cell 6: Check 1 - Full test suite with CUDA
!pytest -x -q


==================================== ERRORS ====================================
______________ ERROR collecting tests/test_opt/test_calibrate.py _______________
ImportError while importing test module '/content/Stackkernels/tests/test_opt/test_calibrate.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_opt/test_calibrate.py:4: in <module>
    from tests.fixtures.profiles import CPU_PROFILE, T4_PROFILE
E   ModuleNotFoundError: No module named 'tests.fixtures'
=========================== short test summary info ============================
ERROR tests/test_opt/test_calibrate.py
!!!!!!!!!!!!!!!!!!!!!!!!!! stopping after 1 failures !!!!!!!!!!!!!!!!!!!!!!!!!!!
1 error in 0.22s


In [26]:
# Cell 7: Check 2 - Triton vs Fallback Coverage
import stackkernels
from stackkernels._options import options
options(backend="triton")
print("Options backend:", options().backend)
!pytest tests/ops/test_rms_norm.py tests/ops/test_layer_norm.py tests/ops/test_glu.py tests/ops/test_rope.py tests/ops/test_cross_entropy.py tests/ops/test_flash_attn.py -v

Options backend: triton
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/Stackkernels
configfile: pyproject.toml
plugins: anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12, platformdirs-4.13.5
collected 213 items                                                            

tests/ops/test_rms_norm.py::test_rms_norm_forward_torch[dtype0-False-shape0] PASSED [  0%]
tests/ops/test_rms_norm.py::test_rms_norm_forward_torch[dtype0-False-shape1] PASSED [  0%]
tests/ops/test_rms_norm.py::test_rms_norm_forward_torch[dtype0-False-shape2] PASSED [  1%]
tests/ops/test_rms_norm.py::test_rms_norm_forward_torch[dtype0-False-shape3] FAILED [  1%]
tests/ops/test_rms_norm.py::test_rms_norm_forward_torch[dtype0-True-shape0] PASSED [  2%]
tests/ops/test_rms_norm.py::test_rms_norm_forward_torch[dtype0-True-shape1] PASSED [  2%]
tests/ops/test_rms_norm.py::

In [27]:
# Cell 8: Check 3 - Device Profile Query
!python -m stackkernels info

=== StackKernels System & Hardware Info ===
StackKernels version: 0.1.0
PyTorch version:      2.11.0+cu130
Triton version:       3.6.0
Active backend:       auto
--- Device Profile ---
Device name:          Tesla T4
Compute capability:   sm_75
SM / Core count:      40
Total memory:         14.56 GB
Has cp.async:         False
Has native BF16:      False
Device fingerprint:   tesla-t4_sm75_40sm_14912mb
--- Calibrated Peaks ---
Status:               measured
Peak bandwidth:       241.26 GB/s
FP16 compute:         26.63 TFLOPs
FP32 compute:         4.3 TFLOPs


In [28]:
# Cell 9: Check 4 - Hardware Calibration
!python -m stackkernels calibrate

Calibrating device 'Tesla T4' (fingerprint: tesla-t4_sm75_40sm_14912mb)...
Calibration complete:
  Achieved Bandwidth: 242.3 GB/s
  FP16 Throughput:    22.04 TFLOPs
  FP32 Throughput:    3.92 TFLOPs


In [29]:
# Cell 10: Check 5 - Probe Adapter on RMSNorm
from stackkernels._core.norm.rms_norm import _rms_norm_fwd_kernel
from stackkernels._opt.occupancy import probe
from stackkernels._opt.hardware import get_device_profile

prof = get_device_profile()
print(f"Device: {prof.name} ({prof.fingerprint()})")
for dtype in ["float16", "float32"]:
    for cols in [1024, 4096, 16384]:
        res = probe(_rms_norm_fwd_kernel, profile=prof, num_warps=4)
        print(f"dtype={dtype:<7} cols={cols:<5} -> {res}")

Device: Tesla T4 (tesla-t4_sm75_40sm_14912mb)
dtype=float16 cols=1024  -> None
dtype=float16 cols=4096  -> None
dtype=float16 cols=16384 -> None
dtype=float32 cols=1024  -> None
dtype=float32 cols=4096  -> None
dtype=float32 cols=16384 -> None


In [30]:
# Cell 11: Check 6 - Resolution Order
from stackkernels._opt.pick import pick
from stackkernels._opt.tune import tune

print("--- Pick BEFORE tuning (Heuristic expected) ---")
cfg1, exp1 = pick("rms_norm", shape=(2048, 1024), dtype="float16", explain=True)
print(exp1)

print("\n--- Executing tune() ---")
tune("rms_norm", shape=(2048, 1024), dtype="float16")

print("\n--- Pick AFTER tuning (User Cache expected) ---")
cfg2, exp2 = pick("rms_norm", shape=(2048, 1024), dtype="float16", explain=True)
print(exp2)

--- Pick BEFORE tuning (Heuristic expected) ---
=== StackKernels Pick Explanation: 'rms_norm' ===
Decision Source: user_cache
Device Profile: Tesla T4 (fingerprint: tesla-t4_sm75_40sm_14912mb)
Triton Version: 3.6.0, Kernel Revision: 1
Hardware Facts:
  - SM Count: 40
  - Max Threads / SM: 1024
  - Max Regs / SM: 65536
  - Max Regs / Thread: 255
  - Max Smem / Block: 49152 bytes
Occupancy & Wave Math:
  - Problem Shape: (2048, 1024), Dtype: float16
  - Grid Size: 2048 program blocks
  - Waves: 26.00 waves across 40 SMs
  - Estimated Occupancy: 2 programs/SM (1024 active threads/SM)
Selected Configuration:
  - {'BLOCK_SIZE': 2048, 'num_warps': 16, 'num_stages': 1, 'variant': 'persistent'}
Rejected Candidates: None

--- Executing tune() ---

--- Pick AFTER tuning (User Cache expected) ---
=== StackKernels Pick Explanation: 'rms_norm' ===
Decision Source: user_cache
Device Profile: Tesla T4 (fingerprint: tesla-t4_sm75_40sm_14912mb)
Triton Version: 3.6.0, Kernel Revision: 1
Hardware Facts:


In [31]:
# Cell 12: Check 7 - Tune One Representative Per Kernel Class
from stackkernels._opt.tune import tune
from stackkernels._opt.hardware import get_device_profile

prof = get_device_profile()
representatives = [
    ("Row (RMSNorm)", "rms_norm", (2048, 1024), "float16"),
    ("Row (CrossEntropy)", "cross_entropy", (2048, 32000), "float16"),
    ("Elementwise (SwiGLU)", "swiglu", (2048, 4096), "float16"),
    ("Tiled (FlashAttention D=64)", "flash_fwd", (2, 8, 1024, 64), "float16"),
    ("Tiled (FlashAttention D=128)", "flash_fwd", (2, 8, 1024, 128), "float16"),
    ("Decode (FlashDecode)", "flash_decode", (2, 8, 1, 64), "float16"),
]

for label, op, shape, dt in representatives:
    print(f"\n=== Tuning {label} ===")
    best = tune(op, shape=shape, dtype=dt, profile=prof)
    print(f"Result: {best}")


=== Tuning Row (RMSNorm) ===
Result: {'BLOCK_SIZE': 1024, 'num_warps': 4, 'num_stages': 1}

=== Tuning Row (CrossEntropy) ===
Result: {'BLOCK_SIZE': 1024, 'num_warps': 8, 'num_stages': 1}

=== Tuning Elementwise (SwiGLU) ===
Result: {'BLOCK_SIZE': 512, 'num_warps': 8, 'num_stages': 1}

=== Tuning Tiled (FlashAttention D=64) ===
Result: {'BLOCK_M': 32, 'BLOCK_N': 64, 'num_warps': 4, 'num_stages': 1, 'kernel_variant': 'flash_fwd'}

=== Tuning Tiled (FlashAttention D=128) ===
Result: {'BLOCK_M': 32, 'BLOCK_N': 32, 'num_warps': 4, 'num_stages': 1, 'kernel_variant': 'flash_fwd'}

=== Tuning Decode (FlashDecode) ===
Result: {'SPLIT_KV': 16, 'BLOCK_M': 16, 'BLOCK_N': 64, 'num_warps': 4, 'num_stages': 1, 'kernel_variant': 'decode'}


In [32]:
# Cell 13: Check 8 - Correctness of Winners
from stackkernels._opt.pick import pick
import stackkernels.torch as s_torch
import stackkernels._reference.torch as ref

print("=== Check 8: Correctness of Tuned Winners vs Reference ===")
device = "cuda" if torch.cuda.is_available() else "cpu"

# RMSNorm
cfg = pick("rms_norm", shape=(2048, 1024))
x = torch.randn(2048, 1024, device=device, dtype=torch.float16, requires_grad=True)
w = torch.randn(1024, device=device, dtype=torch.float16, requires_grad=True)
out_t = s_torch.rms_norm(x, w)
out_r = ref.rms_norm(x, w)
max_err = (out_t - out_r).abs().max().item()
print(f"RMSNorm Winner Max Error: {max_err:.2e}")
assert max_err < 1e-2, f"Correctness failed: {max_err}"

=== Check 8: Correctness of Tuned Winners vs Reference ===
RMSNorm Winner Max Error: 7.81e-03


In [33]:
# Cell 14: Check 9 - Persistent RMSNorm Variant Validation
import torch
from stackkernels._core.norm.rms_norm import rms_norm_fwd
import stackkernels._reference.torch as ref
from stackkernels._opt.profile.bench import bench_fn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("=== Check 9: Persistent RMSNorm Variant ===")
cols_list = [512, 1024, 2048]
rows_list = [64, 161, 1000, 8192, 8193]

print("Part A: Correctness across shapes")
all_passed = True
for dt in [torch.float16, torch.float32]:
    for c in cols_list:
        for r in rows_list:
            x = torch.randn(r, c, device=device, dtype=dt)
            w = torch.randn(c, device=device, dtype=dt)
            # Persistent forward
            y_p, _ = rms_norm_fwd(x, w, config={"variant": "persistent", "BLOCK_SIZE": 1024 if c<=1024 else 2048, "sm_count": 40})
            # Default forward
            y_d, _ = rms_norm_fwd(x, w, config={"variant": "default", "BLOCK_SIZE": 1024 if c<=1024 else 2048})
            # Reference
            y_r = ref.rms_norm(x, w)
            err_p = (y_p - y_r).abs().max().item()
            if err_p > 1e-2:
                print(f"FAIL at dt={dt}, r={r}, c={c}: err={err_p:.2e}")
                all_passed = False

if all_passed:
    print("All persistent variant correctness checks PASSED!")

print("\nPart B: Benchmark persistent vs default at rows=8192")
for dt in [torch.float16, torch.float32]:
    for c in cols_list:
        x = torch.randn(8192, c, device=device, dtype=dt)
        w = torch.randn(c, device=device, dtype=dt)
        cfg_p = {"variant": "persistent", "BLOCK_SIZE": 1024 if c<=1024 else 2048, "sm_count": 40}
        cfg_d = {"variant": "default", "BLOCK_SIZE": 1024 if c<=1024 else 2048}
        res_p = bench_fn(lambda: rms_norm_fwd(x, w, config=cfg_p), warmup=10, iters=30)
        res_d = bench_fn(lambda: rms_norm_fwd(x, w, config=cfg_d), warmup=10, iters=30)
        speedup = (res_d.median_ms - res_p.median_ms) / res_d.median_ms * 100
        print(f"dt={str(dt):<13} cols={c:<4} | Default: {res_d.median_ms:.3f} ms | Persistent: {res_p.median_ms:.3f} ms | Speedup: {speedup:+.1f}%")

=== Check 9: Persistent RMSNorm Variant ===
Part A: Correctness across shapes
All persistent variant correctness checks PASSED!

Part B: Benchmark persistent vs default at rows=8192
dt=torch.float16 cols=512  | Default: 0.039 ms | Persistent: 0.040 ms | Speedup: -4.2%
dt=torch.float16 cols=1024 | Default: 0.043 ms | Persistent: 0.043 ms | Speedup: -0.3%
dt=torch.float16 cols=2048 | Default: 0.041 ms | Persistent: 0.041 ms | Speedup: +1.3%
dt=torch.float32 cols=512  | Default: 0.046 ms | Persistent: 0.040 ms | Speedup: +11.6%
dt=torch.float32 cols=1024 | Default: 0.039 ms | Persistent: 0.045 ms | Speedup: -14.5%
dt=torch.float32 cols=2048 | Default: 0.039 ms | Persistent: 0.040 ms | Speedup: -1.9%


In [34]:
# Cell 15: Check 10 - Cache Export
import os, json
from stackkernels._opt.cache import get_global_cache

cache = get_global_cache()
cache_path = cache.cache_dir / "user_cache.json"
print(f"Cache file path: {cache_path}")
if cache_path.exists():
    with open(cache_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    raw = json.dumps(data, indent=2)
    print("BEGIN_JSON tuned_cache.json")
    print(raw)
    print("END_JSON tuned_cache.json")
    os.makedirs("benchmarks/results", exist_ok=True)
    with open("benchmarks/results/validation_t4.json", "w", encoding="utf-8") as f:
        f.write(raw)
    print("Saved copy to benchmarks/results/validation_t4.json")
else:
    print("No user cache file found on disk.")

Cache file path: /root/.cache/stackkernels/user_cache.json
BEGIN_JSON tuned_cache.json
{
  "rms_norm:tesla-t4_sm75_40sm_14912mb:3.6.0:1:float16:2048x1024": {
    "config": {
      "BLOCK_SIZE": 1024,
      "num_warps": 4,
      "num_stages": 1
    },
    "metadata": {
      "config": {
        "BLOCK_SIZE": 1024,
        "num_warps": 4,
        "num_stages": 1
      },
      "median_ms": 0.1172,
      "achieved_gbps": 71.63,
      "percent_of_peak": 29.56,
      "n_regs": 32,
      "n_spills": 0,
      "programs_per_sm": 4,
      "gain_over_heuristic_pct": 40.05,
      "triton_version": "3.6.0",
      "kernel_rev": 1
    }
  },
  "cross_entropy:tesla-t4_sm75_40sm_14912mb:3.6.0:1:float16:2048x32000": {
    "config": {
      "BLOCK_SIZE": 1024,
      "num_warps": 8,
      "num_stages": 1
    },
    "metadata": {
      "config": {
        "BLOCK_SIZE": 1024,
        "num_warps": 8,
        "num_stages": 1
      },
      "median_ms": 0.1224,
      "achieved_gbps": 2142.88,
      "percent_o